# Persiapan Hari 3 dan 4 (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari3/00-persiapan.ipynb)

Hari 3 dan 4 memakai LangChain dan LangGraph untuk membangun agen: tool, memori, persetujuan manusia, guardrail, multi-agen, MCP, dan mini project. Konsep model bahasa dan RAG sudah dibahas di hari 1 dan 2, jadi tidak diulang di sini.

Seluruh bab dijalankan di **Google Colab**, tanpa instalasi apa pun di laptop. Notebook ini menyiapkan runtime Colab: menyalin repository, memasang pustaka, mengisi API key, lalu menjalankan pemeriksaan. Setiap bab punya notebook Colab sendiri dan memakai tiga cell Persiapan yang sama dengan di bawah, jadi bab berikutnya cukup mengulang tiga cell itu.

## Yang Berbeda dari Versi VS Code

| Pada versi VS Code | Di Colab |
|---|---|
| Python dan virtual environment dipasang sendiri | Disediakan runtime Colab, tidak ada `.venv` |
| File `.env` disalin dan diisi manual | Dibuat otomatis oleh cell Persiapan 2 |
| `pip install -r requirements.txt` di terminal | Cell Persiapan 1 memasang `colab/requirements-colab.txt` dan paket repo (`-e .`) |
| Model embedding tersimpan di folder `models/` | Diunduh (sekitar 220 MB) pada pemakaian pertama di setiap sesi baru |
| Perintah di terminal | Ditulis di cell dengan awalan `!`, misalnya `!python setup_check.py` |
| Model lokal (Ollama) dan LangGraph Studio | Tidak tersedia di Colab; hari 3 dan 4 memakai profil online |

## Langkah 1: Simpan Salinan ke Drive

Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri. Notebook asli tidak dapat diubah.

## Langkah 2 (Opsional): Simpan API Key di Colab Secrets

Tanpa langkah ini, cell Persiapan 2 meminta API key setiap kali Anda membuka notebook baru. Dengan Secrets, key cukup diisi sekali dan berlaku untuk semua notebook di akun Anda:

1. Klik ikon kunci di panel kiri Colab.
2. Pilih **Add new secret**. Buat secret bernama `ONLINE_API_KEY` dan isi dengan API key workshop. Buat juga `PARTICIPANT_ID` (nomor meja dan nama singkat, contoh `P07-andi`).
3. Aktifkan **Notebook access** pada kedua secret.

## Langkah 3: Jalankan Tiga Cell Persiapan

Jalankan cell berikut berurutan dengan `Shift+Enter`. Cell pertama memakan waktu sekitar 2-3 menit pada sesi baru.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Langkah 4: Uji Persiapan Anda

Cell berikut menjalankan `setup_check.py`: memeriksa versi Python, paket, file `.env`, data, satu panggilan ke model, dan model embedding. Unduhan model embedding (sekitar 220 MB) membuat pemeriksaan pertama lebih lama.

In [ ]:
# Pemeriksaan lengkap: Python, paket, .env, data, panggilan model, dan model embedding
# (model embedding sekitar 220 MB diunduh pada pemeriksaan pertama di sesi ini)
periksa = subprocess.run([sys.executable, "setup_check.py"], capture_output=True, text=True)
print(periksa.stdout)
if periksa.returncode != 0:  # stderr hanya ditampilkan bila ada pemeriksaan yang gagal
    print(periksa.stderr[-1500:])

Kalau Anda melihat "Seluruh pemeriksaan berhasil", Anda siap. Baris berawalan `GAGAL` berarti ada yang perlu diperbaiki; lihat Pemecahan Masalah di bawah. Baris berawalan `PERINGATAN` tidak menggagalkan pemeriksaan. Di sesi Colab yang baru, peringatan tentang sisa upload atau index Chroma dari hari 2 tidak akan muncul karena repository disalin segar.

Model embedding hanya dibutuhkan oleh contoh yang memakai tool pencarian dokumen `cari_dokumen` (bab 2, 3, 6, 7, 8, 9, 10, dan 11). Bab 1, 4, 5, dan kedua bab bonus berjalan tanpa embedding.

## Mengerjakan Latihan di Colab

Setiap bab punya satu notebook latihan (`NN-nama-bab-latihan`) yang memuat semua latihan bab itu, berurutan dari atas ke bawah. Buka notebook latihannya, jalankan tiga cell Persiapan, lalu isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab itu (tautan PDF ada di bagian atas notebook). Setelah mengisi, jalankan cell dengan `Shift+Enter`. Solusi lengkap yang tinggal dijalankan ada di folder `colab/solusi/`; pakai untuk mencocokkan hasil atau saat macet.

## Pemecahan Masalah

| Gejala | Penyebab umum | Perbaikan |
|---|---|---|
| Cell Persiapan 1 gagal pada `pip install` | Koneksi terputus saat mengunduh | Jalankan ulang cell itu |
| Muncul pesan untuk me-restart session setelah `pip install` | Colab sudah memuat versi paket yang lebih lama | **Runtime > Restart session**, lalu jalankan ulang dari Persiapan 1 |
| `ModuleNotFoundError: migas_shared` atau `common` | Persiapan 1 belum dijalankan di sesi ini | Jalankan Persiapan 1, lalu cell yang gagal |
| `401` atau `Unauthorized` | API key salah atau ada spasi tambahan | Hapus `/content/workshop-agent-migas/.env` lewat panel Files, lalu jalankan ulang Persiapan 2 |
| `Connection error` | Koneksi internet runtime terputus | Cek koneksi, lalu jalankan ulang cell |
| Runtime terputus atau semua file hilang | Colab menghapus `/content` setelah sesi tidak aktif | Jalankan ulang Persiapan 1 sampai 3, lalu lanjutkan dari cell yang sedang dikerjakan |
| Unduhan model embedding gagal atau lambat | Jaringan terbatas | Jalankan ulang cell; unduhan dilanjutkan |
| `GAGAL Data workshop` | Folder `data/` tidak lengkap karena `git clone` terputus | Hapus folder `workshop-agent-migas` lewat panel Files, lalu jalankan ulang Persiapan 1 |

## Langkah Selanjutnya

Lanjut ke [Dasar LangChain](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari3/01-dasar-langchain.ipynb).